# Clase 14 — Del agente que construimos a una aplicación web
**AI Agent Developer · ITBA Educación Ejecutiva**  
**Laboratorio incremental — C11 + C12 + C13 → FastAPI + Swagger + Streamlit + Docker Compose**

Un objetivo concreto: que otra persona pueda utilizar **las capacidades que ya construimos** sin abrir la notebook. Al terminar podrás ejecutar la misma operación desde Python, desde Swagger y desde Streamlit, y comprobar que los tres clientes observan el mismo *run*.

> **Alcance del caso:** usamos correos sintéticos, la búsqueda local de la clase 13 y un borrador determinístico; el sistema no envía emails ni ejecuta Gmail real. El motor de transiciones de `hitl.py` (C11) y el orquestador `multiagent.py` (C12) se reutilizan tal como quedaron persistidos.

## Ruta de la práctica: seis paradas visibles

| Parada | ¿Qué vas a comprobar? | Dónde lo observás |
|---|---|---|
| **A — HTTP** | La API publica operaciones y devuelve `201`/`200` | Swagger `/docs` |
| **B — Estado** | Un `run_id` sobrevive a recargar el navegador | Swagger y checkpoint |
| **C — HITL** | Editar crea otra versión y requiere nueva aprobación | Swagger |
| **D — Eventos** | JSON de eventos y replay SSE no son lo mismo que streaming en vivo | `curl -N` y Swagger |
| **E — UI** | Swagger y Streamlit muestran el mismo run | Dos pestañas |
| **F — Docker** | La aplicación funciona fuera de Jupyter; un volumen conserva los runs | Docker Compose |

**Criterio de éxito:** no memorizar comandos, sino poder indicar qué capa es responsable de cada resultado y demostrarlo.

## 0. Preparación y continuidad C11–C13

En la carpeta desde la que abrís esta notebook debe existir `ai_agent_project/` con los módulos de las clases anteriores. Si estás usando el paquete distribuido para esta clase, ese directorio ya viene incluido como **base de referencia**; si tenés tu proyecto incremental, usá ese mismo y no lo sobrescribas.

La notebook **solo escribe archivos nuevos de Clase 14** (`class14_*.py`, `streamlit_app.py`, Dockerfile y Compose). No modifica `hitl.py`, `multiagent.py`, `support_mcp_bridge.py` ni los datos del caso.

Para instalar las dependencias, abrí una terminal y ejecutá, desde el proyecto:

```bash
cd ai_agent_project
python -m pip install -r requirements-class14.txt
```

**Importante:** la instalación requiere acceso a Internet; los tests de C14 no llaman a modelos pagos ni requieren credenciales.

In [ ]:
from pathlib import Path
import sys, json, asyncio, importlib, inspect

PROJECT_DIR = Path("ai_agent_project").resolve()
SRC_DIR = PROJECT_DIR / "src"
REQUIRED = [
    "src/ai_agent_course/hitl.py",          # Estados y HITL de C11
    "src/ai_agent_course/multiagent.py",    # Orquestador C12
    "src/ai_agent_course/support_mcp_bridge.py",  # Búsqueda local C13
    "src/ai_agent_course/support_case.py",  # Loader del caso sintético
    "data/mock_gmail/threads.json",
    "data/mock_gmail/knowledge_base.md",
]
missing = [path for path in REQUIRED if not (PROJECT_DIR / path).is_file()]
if missing:
    raise FileNotFoundError("Faltan prerrequisitos C11–C13: " + ", ".join(missing))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print("Proyecto:", PROJECT_DIR)
print("Continuidad: archivos presentes ✓")

### 0.1 Verificar contratos, no solamente nombres de archivos

Dos estudiantes pueden tener un archivo `hitl.py` con implementaciones de distintas semanas. Por eso verificamos también que existan las funciones concretas que usa la clase. Si esta comprobación falla, recuperá los módulos **actualizados** de las notebooks 11–13: el laboratorio no los recreará silenciosamente.

In [ ]:
from ai_agent_course import hitl, multiagent, support_mcp_bridge, support_case

expected = {
    hitl: ["AgentRunState", "JsonCheckpointStore", "prepare_action",
           "apply_human_decision", "mark_policy_authorized", "canonical_hash"],
    multiagent: ["WorkerTask", "WorkerOutput", "Evidence", "run_supervisor_workers"],
    support_mcp_bridge: ["search_support_kb"],
    support_case: ["load_support_case"],
}
for module, names in expected.items():
    absent = [name for name in names if not hasattr(module, name)]
    if absent:
        raise RuntimeError(f"{module.__name__} es de otra versión; faltan {absent}. "
                           "Ejecutá nuevamente las celdas de persistencia de C11–C13.")
print("Contratos C11–C13 compatibles ✓")

### 0.2 El mismo correo y la misma búsqueda de conocimiento

Elegí uno de los correos sintéticos. La capacidad de búsqueda ya existe: en la clase 13 fue expuesta mediante una herramienta MCP, pero aquí reutilizamos **la función local** `search_support_kb`, sin levantar un servidor MCP.

In [ ]:
threads, _ = support_case.load_support_case(PROJECT_DIR)
for t in threads:
    print(t["thread_id"], "|", t["subject"])
selected_thread = threads[0]
sample = support_mcp_bridge.search_support_kb(
    PROJECT_DIR,
    selected_thread["subject"] + " " + selected_thread["messages"][0]["body"],
    top_k=2,
)
print("Fuentes recuperadas:", [f"{h.source}#{h.locator}" for h in sample.hits])

## 1. La frontera de aplicación: reutilizar el motor existente

La arquitectura es **cliente → transporte HTTP → servicio adaptador → C11–C13**. Una función que resuelve un problema del dominio no debe depender de FastAPI. Por eso primero construimos un adaptador invocable **directamente desde Python**; después lo publicamos en HTTP.

El siguiente archivo conecta los *workers* y el retriever existentes. La salida contiene un borrador, evidencia y trazas. **No agrega otro orquestador**.

### 1.1 Adaptador de capacidades C12/C13

**Entrada:** caso sintético. **Proceso:** triage y búsqueda de evidencia; writer depende de ambos. **Salida:** borrador y fuentes. 

Leé especialmente la llamada a `run_supervisor_workers`: viene de **C12**. La búsqueda `search_support_kb` viene de **C13**. El borrador es deliberadamente determinístico: el objetivo de C14 es publicar el sistema, no crear un nuevo agente.

In [ ]:
%%writefile ai_agent_project/src/ai_agent_course/class14_support_flow.py
"""Adaptador C14: invoca el orquestador C12 y la búsqueda local de C13.

No crea otro sistema multiagente ni un proveedor de correo; todos los casos
utilizan datos sintéticos de ``data/mock_gmail``. El resultado es un borrador
que todavía requiere supervisión humana antes de completar el dry-run.
"""
from __future__ import annotations
import asyncio
from ai_agent_course.multiagent import Evidence, WorkerTask, WorkerOutput, run_supervisor_workers
from ai_agent_course.support_mcp_bridge import search_support_kb


def thread_text(thread: dict) -> str:
    """Devuelve asunto + mensajes del hilo, utilizados como consulta de soporte."""
    return thread["subject"] + ". " + " ".join(str(m["body"]) for m in thread["messages"])


async def prepare_support_draft(project_dir, thread: dict) -> dict:
    """Coordina triage + knowledge y prepara un borrador con trazabilidad.

    Args:
        project_dir: Raíz del proyecto, que contiene ``data/mock_gmail``.
        thread: Correo sintético seleccionado por el alumno.
    Returns:
        Diccionario con draft, evidence, trace y origen de las capacidades.
    """
    text = thread_text(thread)

    async def triage(task: WorkerTask, _deps: list[WorkerOutput]) -> WorkerOutput:
        urgent = any(x in text.lower() for x in ("bloquea", "cierre mensual", "urgente"))
        return WorkerOutput(task_id=task.task_id, role="triage", status="success",
                            summary=f"intent=soporte; urgency={'alta' if urgent else 'media'}",
                            confidence=0.9)

    async def knowledge(task: WorkerTask, _deps: list[WorkerOutput]) -> WorkerOutput:
        # La búsqueda de C13 es síncrona; no bloqueamos el loop de C12.
        result = await asyncio.to_thread(search_support_kb, project_dir, text, top_k=2)
        if not result.hits:
            return WorkerOutput(task_id=task.task_id, role="knowledge", status="partial",
                                summary="No hay evidencia en la base", confidence=0,
                                open_questions=["Escalar: falta información respaldada"])
        hit = result.hits[0]
        return WorkerOutput(task_id=task.task_id, role="knowledge", status="success",
                            summary=hit.content, confidence=0.85,
                            evidence=[Evidence(claim=hit.content, source=hit.source,
                                               locator=hit.locator)])

    async def writer(task: WorkerTask, deps: list[WorkerOutput]) -> WorkerOutput:
        kb = next((out for out in deps if out.role == "knowledge"), None)
        evidence = list(kb.evidence) if kb else []
        if not kb or kb.status != "success" or not evidence:
            draft = "No hay evidencia suficiente. Escalaremos este caso antes de responder."
            status = "partial"
        else:
            ref = evidence[0]
            draft = (f"Gracias por escribirnos. Según {ref.source}, sección {ref.locator}, "
                     "preparamos una propuesta para revisión humana. "
                     "Este borrador todavía no se envió.")
            status = "success"
        return WorkerOutput(task_id=task.task_id, role="writer", status=status,
                            summary=draft, evidence=evidence, confidence=0.8 if evidence else 0,
                            assumptions=["requires_human=true"])

    plan = [
        WorkerTask(task_id="triage", role="triage", objective="Clasificar la urgencia del correo",
                   success_criteria=["urgencia identificada"]),
        WorkerTask(task_id="knowledge", role="knowledge", objective="Buscar respaldo en C13",
                   success_criteria=["evidencia o abstención explícita"]),
        WorkerTask(task_id="writer", role="writer", objective="Preparar borrador para revisión",
                   success_criteria=["borrador o abstención"], depends_on=["triage", "knowledge"]),
    ]
    # Aquí se llama literalmente a la función que se persistió en la clase 12.
    result = await run_supervisor_workers(
        plan, {"triage": triage, "knowledge": knowledge, "writer": writer},
        max_concurrency=2, timeout_s=8,
    )
    output = result["outputs"]["writer"]
    return {
        "draft": output.summary,
        "status": output.status,
        "evidence": [e.model_dump() for e in output.evidence],
        "trace": result["trace"],
        "engine": "C12.run_supervisor_workers",
        "retrieval": "C13.search_support_kb (modo local; no transporte MCP remoto)",
    }

### Comprobación 1 — La capacidad funciona sin API

Si este resultado falla, no tiene sentido abrir Swagger todavía. La frontera HTTP debe envolver una capacidad existente que ya funciona.

In [ ]:
importlib.invalidate_caches()
from ai_agent_course.class14_support_flow import prepare_support_draft

prepared = await prepare_support_draft(PROJECT_DIR, selected_thread)
assert prepared["engine"].startswith("C12")
assert prepared["retrieval"].startswith("C13")
assert prepared["draft"]
print("Borrador:", prepared["draft"])
print("Evidencia:", prepared["evidence"])
print("Traza:", [row.get("event") for row in prepared["trace"]])

## 2. Servicio de aplicación: operaciones sobre el mismo State C11

El **servicio adaptador** ofrece operaciones orientadas al caso de uso (`start`, `read`, `decide`, `resume`, `events`). Reutiliza `AgentRunState` y las transiciones de `hitl.py`; **no inventa otra máquina de estados**.

La persistencia de esta práctica es `JsonCheckpointStore` de C11: permite recuperar una ejecución entre requests o después de reiniciar **un proceso**. No es una base de datos apta para múltiples workers concurrentes.

### 2.1 Una sola fuente de verdad

Prestá atención a estas tres líneas en el módulo: `prepare_action(...)`, `apply_human_decision(...)` y `mark_policy_authorized(...)`. Son **funciones de C11**. El servicio solo las organiza para que puedan invocarse desde HTTP. 

`proposal_version` y `args_hash` protegen la aprobación: un permiso emitido para la versión 1 no autoriza automáticamente la versión 2.

In [ ]:
%%writefile ai_agent_project/src/ai_agent_course/class14_service.py
"""Servicio de aplicación C14: orquesta operaciones del dominio existente.

``hitl.py`` es la fuente de verdad para estados, versiones y hash. La clase 14
solo añade una fachada para los clientes HTTP. El JSON es didáctico: la
sincronización sirve únicamente para un proceso y un worker.
"""
from __future__ import annotations
import threading
from pathlib import Path
from pydantic import BaseModel, ConfigDict, Field
from ai_agent_course.hitl import (
    AgentRunState, JsonCheckpointStore, add_event, apply_human_decision,
    canonical_hash, mark_policy_authorized, prepare_action,
)
from ai_agent_course.support_case import load_support_case
from ai_agent_course.class14_support_flow import prepare_support_draft


class DraftArguments(BaseModel):
    """Contrato de la única acción permitida: guardar un borrador de prueba."""
    model_config = ConfigDict(extra="forbid", strict=True)
    recipient: str = Field(pattern=r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
    subject: str = Field(min_length=3, max_length=200)
    body: str = Field(min_length=10, max_length=5000)


class CourseApplication:
    """Fachada común utilizada por FastAPI, pruebas y clientes externos.

    Args:
        project_dir: Proyecto C11–C13 con el caso de soporte sintético.
        store_path: Ruta al JSON de checkpoints de C11, fuera del código fuente.
    """
    def __init__(self, project_dir: str | Path, store_path: str | Path):
        self.project_dir = Path(project_dir).resolve()
        self.store = JsonCheckpointStore(store_path)
        # RLock no reemplaza una base transaccional entre procesos distintos.
        self.lock = threading.RLock()

    def cases(self) -> list[dict]:
        """Lista los correos sintéticos disponibles para elegir desde la UI."""
        threads, _ = load_support_case(self.project_dir)
        return [{"thread_id": t["thread_id"], "subject": t["subject"]} for t in threads]

    async def start(self, thread_id: str, user_message: str) -> AgentRunState:
        """Crea un run nuevo y lo deja detenido en el HITL original de C11."""
        threads, _ = load_support_case(self.project_dir)
        thread = next((t for t in threads if t["thread_id"] == thread_id), None)
        if thread is None:
            raise KeyError(f"Caso desconocido: {thread_id}")
        result = await prepare_support_draft(self.project_dir, thread)
        state = AgentRunState(thread_id=thread_id, user_message=user_message,
                              context_capsule={"class14_support": result})
        state = prepare_action(
            state, tool_name="create_draft",
            args={"recipient": "cliente@example.com", "subject": thread["subject"],
                  "body": result["draft"]},
            rationale="Borrador respaldado por C12/C13 pendiente de aprobación",
            evidence_used=[f"{e['source']}#{e['locator']}" for e in result["evidence"]],
            risk_level="medium",
        )
        with self.lock:
            return self.store.save(state)

    def read(self, run_id: str) -> AgentRunState:
        """Recupera el checkpoint persistido por run_id."""
        with self.lock:
            return self.store.load(run_id)

    def decide(self, run_id: str, payload: dict) -> AgentRunState:
        """Aplica approve/edit/reject VALIDANDO la versión y el hash de C11."""
        with self.lock:
            state = self.store.load(run_id)
            state = apply_human_decision(state, payload,
                                         editable_fields={"subject", "body"})
            return self.store.save(state)

    def resume(self, run_id: str) -> AgentRunState:
        """Revalida la aprobación y finaliza el dry-run sin enviar correos."""
        with self.lock:
            state = self.store.load(run_id)
            if state.status != "approved_pending_policy":
                raise ValueError("El run no está aprobado o ya terminó")
            action, decision = state.pending_action, state.last_decision
            if not (action and decision and decision.decision == "approve"
                    and action.action_id == decision.action_id
                    and action.version == decision.proposal_version
                    and action.args_hash == decision.args_hash
                    and canonical_hash(action.args) == action.args_hash):
                raise ValueError("Los argumentos ya no coinciden con lo aprobado")
            # Control local acotado a create_draft. No se atribuye al gate de C9.
            if action.tool_name != "create_draft":
                raise ValueError("Sólo está permitida la acción create_draft")
            if action.args.get("recipient") != "cliente@example.com":
                raise ValueError("No se puede modificar el destinatario de prueba")
            validated_args = DraftArguments.model_validate(action.args).model_dump()
            if canonical_hash(validated_args) != action.args_hash:
                raise ValueError("La validación cambió los argumentos aprobados")
            state = mark_policy_authorized(state, validated_args=validated_args)
            add_event(state, "dry_run_completed", email_sent=False)
            state.result = {"status": "completed", "email_sent": False,
                            "draft": dict(action.args), "note": "Dry-run; sin Gmail"}
            state.status = "completed"  # Valor terminal ya definido en C11.
            return self.store.save(state)

    def events(self, run_id: str) -> list[dict]:
        """Devuelve los eventos del run; replay, no publicación en vivo."""
        return self.read(run_id).events

### Comprobación 2 — Iniciar, editar, aprobar y reanudar desde Python

Esta prueba crea un run exclusivo de notebook en `runtime/c14_notebook_checkpoints.json`, separado de los que usarás desde el navegador. Por eso no se generan falsos conflictos al repetir las demostraciones en Swagger.

In [ ]:
from ai_agent_course.class14_service import CourseApplication

service_py = CourseApplication(PROJECT_DIR, PROJECT_DIR / "runtime" / "c14_notebook_checkpoints.json")
run_py = await service_py.start(selected_thread["thread_id"], "Preparar borrador")
assert run_py.status == "waiting_for_approval"
print({"run_id": run_py.run_id, "thread_id": run_py.thread_id,
       "status": run_py.status, "version": run_py.pending_action.version})

In [ ]:
# Usamos el contrato de aprobación de C11 (identidad + versión + hash).
a1 = run_py.pending_action
edit = service_py.decide(run_py.run_id, {
    "decision": "edit", "actor": "alumno-c14", "action_id": a1.action_id,
    "proposal_version": a1.version, "args_hash": a1.args_hash,
    "edited_args": {"body": "Gracias por contactarnos. Verificamos el caso y lo derivaremos a soporte nivel 2."},
})
assert edit.status == "waiting_for_approval"
assert edit.pending_action.version == a1.version + 1
print("La edición creó la propuesta v", edit.pending_action.version)

In [ ]:
# La aprobación posterior tiene que citar la NUEVA versión y hash.
a2 = edit.pending_action
approved = service_py.decide(edit.run_id, {
    "decision": "approve", "actor": "alumno-c14", "action_id": a2.action_id,
    "proposal_version": a2.version, "args_hash": a2.args_hash,
})
assert approved.status == "approved_pending_policy"
completed = service_py.resume(edit.run_id)
assert completed.status == "completed"
assert completed.result["email_sent"] is False
print("Estado final:", completed.status, "| Enviado:", completed.result["email_sent"])

### Comprobación 3 — Persistencia no es estado del navegador

Creá un objeto de servicio nuevo usando la misma ruta de checkpoint. Aunque el objeto anterior deje de existir, el run sigue recuperándose por su identificador.

In [ ]:
service_reopened = CourseApplication(PROJECT_DIR, PROJECT_DIR / "runtime" / "c14_notebook_checkpoints.json")
recovered = service_reopened.read(completed.run_id)
assert recovered.status == "completed"
print("Run recuperado:", recovered.run_id, recovered.status)
print("Eventos:", [x["event"] for x in recovered.events][-5:])

## 3. Publicar operaciones con FastAPI

HTTP es el **transporte**: decide cómo se invoca la aplicación, qué JSON se valida y qué código de estado vuelve. **El servicio** decide qué operación es posible y **C11** valida las transiciones.

Por ejemplo, `@app.post("/runs")` registra una función que atiende requests `POST /runs`. `HTTPException(409)` expresa un conflicto de dominio (p. ej., aprobar una versión vieja); un dato mal formado se rechaza con `422` antes de entrar al servicio.

### 3.1 Endpoints y códigos HTTP

| Ruta | Método | Qué representa |
|---|---|---|
| `/health/live` / `/health/ready` | GET | Proceso disponible / dependencias listas |
| `/cases` | GET | Correos sintéticos entre los que elegir |
| `/runs` | POST | Nueva ejecución (`201`) |
| `/runs/{run_id}` | GET | Estado y propuesta (`200` o `404`) |
| `/runs/{run_id}/decisions` | POST | Aprobar, editar o rechazar (`409` si el State no lo admite) |
| `/runs/{run_id}/resume` | POST | Reanudar una propuesta aprobada |
| `/runs/{run_id}/events` | GET | Historial JSON |
| `/runs/{run_id}/events/stream` | GET | Replay del historial como SSE |

El decorador no contiene la lógica de C11–C13: llama a `service`.

In [ ]:
%%writefile ai_agent_project/src/ai_agent_course/class14_api.py
"""FastAPI C14: traducción de HTTP a las operaciones del servicio de aplicación."""
from __future__ import annotations
import json
import os
from pathlib import Path
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from ai_agent_course.hitl import DecisionMismatchError, InvalidTransitionError
from ai_agent_course.class14_service import CourseApplication

PROJECT_DIR = Path(os.getenv("C14_PROJECT_DIR", Path(__file__).resolve().parents[2]))
STORE_PATH = Path(os.getenv("C14_STORE_PATH", PROJECT_DIR / "runtime" / "class14_runs.json"))
service = CourseApplication(PROJECT_DIR, STORE_PATH)
app = FastAPI(title="Clase 14 - API del agente incremental", version="1.0",
              description="C11+C12+C13 publicados como aplicación. Dry-run, sin Gmail real.")


class StartRun(BaseModel):
    thread_id: str = Field(min_length=1)
    user_message: str = Field(default="Preparar una respuesta con evidencia", min_length=3)


class HumanDecisionIn(BaseModel):
    decision: str = Field(pattern="^(approve|edit|reject)$")
    actor: str = Field(min_length=2)
    action_id: str
    proposal_version: int = Field(ge=1)
    args_hash: str
    edited_args: dict | None = None
    comment: str = ""


def to_http_error(exc: Exception) -> HTTPException:
    """Traduce errores de dominio: inexistente = 404, conflicto = 409."""
    if isinstance(exc, KeyError):
        return HTTPException(status_code=404, detail="No existe el recurso")
    if isinstance(exc, (ValueError, InvalidTransitionError, DecisionMismatchError)):
        return HTTPException(status_code=409, detail=str(exc))
    return HTTPException(status_code=500, detail="Error interno")

#Decoradaro de Fast API
@app.get("/health/live", tags=["salud"])
def health_live():
    return {"status": "ok"}


@app.get("/health/ready", tags=["salud"])
def health_ready():
    try:
        return {"status": "ready", "cases": len(service.cases())}
    except (FileNotFoundError, OSError):
        raise HTTPException(status_code=503, detail="No hay corpus disponible")


@app.get("/cases", tags=["casos"])
def cases():
    return service.cases()


@app.post("/runs", status_code=201, tags=["ejecuciones"])
async def start_run(payload: StartRun):
    try:
        return (await service.start(payload.thread_id, payload.user_message)).model_dump()
    except (KeyError, ValueError, InvalidTransitionError) as exc:
        raise to_http_error(exc) from exc


@app.get("/runs/{run_id}", tags=["ejecuciones"])
def get_run(run_id: str):
    try:
        return service.read(run_id).model_dump()
    except KeyError as exc:
        raise to_http_error(exc) from exc


@app.post("/runs/{run_id}/decisions", tags=["supervision"])
def decision(run_id: str, payload: HumanDecisionIn):
    try:
        return service.decide(run_id, payload.model_dump()).model_dump()
    except (KeyError, ValueError, InvalidTransitionError, DecisionMismatchError) as exc:
        raise to_http_error(exc) from exc


@app.post("/runs/{run_id}/resume", tags=["supervision"])
def resume(run_id: str):
    try:
        return service.resume(run_id).model_dump()
    except (KeyError, ValueError, InvalidTransitionError) as exc:
        raise to_http_error(exc) from exc


@app.get("/runs/{run_id}/events", tags=["observación"])
def events(run_id: str):
    try:
        return service.events(run_id)
    except KeyError as exc:
        raise to_http_error(exc) from exc


@app.get("/runs/{run_id}/events/stream", tags=["observación"])
def replay_events(run_id: str):
    """Serializa eventos YA PERSISTIDOS como SSE; el stream termina."""
    try:
        snapshot = service.events(run_id)
    except KeyError as exc:
        raise to_http_error(exc) from exc

    def encode_events():
        for event in snapshot:
            yield ("event: " + event["event"] + "\n" + "data: " +
                   json.dumps(event, ensure_ascii=False) + "\n\n")

    return StreamingResponse(encode_events(), media_type="text/event-stream")

### Comprobación 4 — Probar el contrato sin levantar servidor

`TestClient` permite verificar la capa HTTP sin abrir un puerto. Usamos un store de pruebas exclusivo para no modificar los runs que crees manualmente desde Swagger.

In [ ]:
from fastapi.testclient import TestClient
from ai_agent_course.class14_api import app, service as http_service
from ai_agent_course.hitl import JsonCheckpointStore

http_service.store = JsonCheckpointStore(PROJECT_DIR / "runtime" / "c14_http_tests.json")
client = TestClient(app)
assert client.get("/health/live").json()["status"] == "ok"
cases_http = client.get("/cases")
assert cases_http.status_code == 200 and cases_http.json()
created = client.post("/runs", json={"thread_id": selected_thread["thread_id"],
                                     "user_message": "Preparar respuesta de prueba"})
assert created.status_code == 201, created.text
run_http = created.json()
assert run_http["status"] == "waiting_for_approval"
print("HTTP:", created.status_code, "run_id:", run_http["run_id"])

### Comprobación 5 — Errores también son parte del contrato

Estos errores no son fallas del sistema: indican que la API protege las invariantes. Cada test crea su propio run o consulta uno inexistente, sin interferir con Swagger.

In [ ]:
assert client.post("/runs", json={"thread_id": ""}).status_code == 422
assert client.get("/runs/no-existe").status_code == 404
# No se puede reanudar sin aprobación humana.
assert client.post(f"/runs/{run_http['run_id']}/resume").status_code == 409
print("422 inválido ✓ | 404 inexistente ✓ | 409 transición imposible ✓")

### Comprobación 6 — El contrato HTTP también completa el ciclo HITL

Ya verificamos los errores `422`, `404` y `409`. Ahora completamos un **segundo run de prueba** desde el cliente HTTP: edición de la propuesta, rechazo de una aprobación con versión antigua, aprobación correcta, reanudación y lectura de eventos como JSON y SSE.

Esta prueba no usa los identificadores creados manualmente en Swagger ni realiza envíos de correo. Los resultados se incorporarán al reporte final de la clase.

In [ ]:
# Contrato completo desde HTTP: edit → stale 409 → approve → resume → events.
# run_http pertenece solo a TestClient; la demo manual en Swagger crea sus propios runs.
http_run_id = run_http["run_id"]
first_action = run_http["pending_action"]
first_approval = {
    "decision": "approve", "actor": "alumno-c14",
    "action_id": first_action["action_id"],
    "proposal_version": first_action["version"],
    "args_hash": first_action["args_hash"],
}
edit_response = client.post(
    f"/runs/{http_run_id}/decisions",
    json={
        **first_approval, "decision": "edit",
        "edited_args": {"body": "Gracias por escribirnos. Revisamos la evidencia y elevaremos el caso a soporte nivel 2."},
    },
)
assert edit_response.status_code == 200, edit_response.text
edited_http = edit_response.json()
new_action = edited_http["pending_action"]
assert new_action["version"] == first_action["version"] + 1

# La decisión para la propuesta original ya es obsoleta.
stale_response = client.post(f"/runs/{http_run_id}/decisions", json=first_approval)
assert stale_response.status_code == 409, stale_response.text

latest_approval = {
    **first_approval,
    "proposal_version": new_action["version"],
    "args_hash": new_action["args_hash"],
}
accepted_response = client.post(f"/runs/{http_run_id}/decisions", json=latest_approval)
assert accepted_response.status_code == 200, accepted_response.text
assert accepted_response.json()["status"] == "approved_pending_policy"
finished_response = client.post(f"/runs/{http_run_id}/resume")
assert finished_response.status_code == 200, finished_response.text
finished_http = finished_response.json()
assert finished_http["status"] == "completed"
assert finished_http["result"]["email_sent"] is False

http_events_response = client.get(f"/runs/{http_run_id}/events")
sse_response = client.get(f"/runs/{http_run_id}/events/stream")
assert http_events_response.status_code == 200
assert sse_response.status_code == 200
http_events = http_events_response.json()
assert http_events and sse_response.text.count("data: ") == len(http_events)

# JSON legible: mostramos estados y evidencias, no el cuerpo completo de mensajes.
http_cycle_summary = {
    "run_id": http_run_id,
    "status_final": finished_http["status"],
    "proposal_version_inicial": first_action["version"],
    "proposal_version_final": new_action["version"],
    "aprobacion_obsoleta_http": stale_response.status_code,
    "eventos_json": len(http_events),
    "eventos_sse_reproducidos": sse_response.text.count("data: "),
    "email_sent": finished_http["result"]["email_sent"],
}
print(json.dumps(http_cycle_summary, ensure_ascii=False, indent=2))

## 4. Llevar la API al navegador — Paradas A y B

**Primera terminal:** situate en `ai_agent_project` y ejecutá:

```bash
python -m uvicorn ai_agent_course.class14_api:app --app-dir src --host 127.0.0.1 --port 8000
```

Mantené la terminal abierta. Luego abrí **http://localhost:8000/docs**. Ese sitio es Swagger UI, generado automáticamente a partir del contrato OpenAPI publicado en `/openapi.json`.

> Al levantar Uvicorn comienza un proceso nuevo y se inicializa su propio `CourseApplication`. Por eso **los runs de Swagger no comparten el store de TestClient**, que sólo se cambió dentro del kernel.

### PARADA A — Explorar y crear una ejecución en Swagger

1. Abrí `GET /cases` → **Try it out** → **Execute**.
2. Elegí el `thread_id` de un caso. Abrí `POST /runs` y ejecutá un JSON como:

```json
{"thread_id": "thr_001", "user_message": "Preparar una respuesta con evidencia"}
```

3. Comprobá que responde **201 Created**, copiá el `run_id` y leé `status`, `pending_action`, `evidence_used` y `args_hash`.
4. Indicá cuál de esas propiedades viene del motor C11 y cuál aporta el adaptador C14.

**Resultado esperado:** `waiting_for_approval`. **No** se envió ningún correo.

### PARADA B — Comprobar continuidad sin notebook

1. Consultá `GET /runs/{run_id}` con el run recién creado.
2. Recargá `/docs` o cerrá y abrí la pestaña del navegador.
3. Volvé a consultar el mismo `run_id`.

**Resultado esperado:** la propuesta y su versión siguen disponibles porque el estado está en `runtime/class14_runs.json`, no en el navegador. `thread_id` identifica el caso; `run_id` identifica el intento de ejecución.

## 5. HITL mediante HTTP — Parada C

El dominio ya sabe editar, aprobar o rechazar. Lo nuevo es que ahora **una persona puede enviar la decisión desde otra aplicación**. Para una aprobación válida necesitás `action_id`, `proposal_version` y `args_hash` de la propuesta actual.

**Usá un run nuevo para esta parada**. No reaproveches el del test automático. Así podrás reconocer conflictos deliberados sin mezclar ejecuciones.

### PARADA C — Una propuesta, dos versiones

1. `POST /runs`: creá un run nuevo. Copiá el `run_id`.
2. `GET /runs/{run_id}`: copiá `action_id`, `version` y `args_hash` de `pending_action`.
3. `POST /runs/{run_id}/decisions`: enviá `decision: "edit"`, `actor: "alumno-c14"`, los campos copiados y `edited_args: {"body": "Respuesta revisada y escalada a nivel 2."}`.
4. `GET /runs/{run_id}`: observá que sigue en `waiting_for_approval` pero la versión aumentó.
5. **Prueba de seguridad:** intentá `approve` usando la versión/hash anteriores → esperá **409 Conflict**.
6. Enviá `approve` con la **versión y hash actuales** → `approved_pending_policy`.
7. `POST /runs/{run_id}/resume` → `completed`, `email_sent: false`.

**Pregunta:** ¿por qué editar una propuesta no equivale a aprobarla?

## 6. Eventos y replay SSE — Parada D

Las clases anteriores incorporaron trazas y eventos. Ahora los publicamos en dos formatos: JSON en `/events` y **Server-Sent Events (SSE)** en `/events/stream`.

Este endpoint **reproduce eventos ya almacenados y termina**. No transmite nuevos eventos en tiempo real. Swagger suele esperar al cierre del stream para presentar la respuesta; `curl -N` permite ver los bloques `event:` y `data:`.

### PARADA D — Dos representaciones del mismo historial

1. Consultá `GET /runs/{run_id}/events`: observá una lista JSON.
2. En la terminal, reemplazá el ID y ejecutá:

```bash
curl -N http://localhost:8000/runs/TU_RUN_ID/events/stream  ó curl.exe -N http://localhost:8000/runs/TU_RUN_ID/events/stream  
```

3. Identificá los encabezados SSE (`event:` y `data:`) y comprobá que el stream finaliza tras reproducir el historial.
4. Relacioná `approval_requested`, `human_decision` y `dry_run_completed` con las operaciones anteriores.

**Pregunta:** ¿qué tendríamos que incorporar para emitir eventos futuros durante una ejecución todavía activa?

## 7. Streamlit — Otra interfaz para el MISMO agente

La interfaz mantiene campos y botones, pero **no importa `hitl.py`, no lee `class14_runs.json` y no ejecuta el agente**. Hace requests HTTP a FastAPI. Es la prueba de que el contrato permite cambiar de cliente sin reescribir el dominio.

### 7.1 Construir un cliente HTTP con Streamlit

Buscá la función `call_api(...)`: encapsula las llamadas con `requests` y muestra un error si la API no responde. `st.session_state` guarda temporalmente el ID seleccionado **en la interfaz**, mientras que el estado durable del run permanece del lado del servidor.

In [ ]:
%%writefile ai_agent_project/streamlit_app.py
"""Interfaz Streamlit: cliente HTTP de FastAPI, sin importar módulos del agente."""
from __future__ import annotations
import os
import requests
import streamlit as st

API = os.getenv("C14_API_URL", "http://localhost:8000").rstrip("/")
st.set_page_config(page_title="Clase 14 · Agente de soporte", page_icon="📨", layout="wide")
st.title("Agente de soporte — Clase 14")
st.caption("Dos clientes, un servicio. Dry-run: no se envía ningún email.")


def call_api(method: str, path: str, **kwargs):
    """Único punto de comunicación; no lee archivos JSON ni ejecuta hitl.py."""
    try:
        response = requests.request(method, API + path, timeout=20, **kwargs)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as exc:
        st.error(f"No pudimos completar la operación en {path}: {exc}")
        return None


cases = call_api("GET", "/cases")
if not cases:
    st.stop()
options = {f'{x["subject"]} · {x["thread_id"]}': x["thread_id"] for x in cases}
selected = st.selectbox("Caso sintético", options)
message = st.text_input("Instrucción", "Preparar respuesta con evidencia y pedir aprobación")
if "c14_run_id" not in st.session_state:
    st.session_state.c14_run_id = ""

if st.button("Crear ejecución", type="primary"):
    result = call_api("POST", "/runs", json={
        "thread_id": options[selected], "user_message": message,
    })
    if result:
        st.session_state.c14_run_id = result["run_id"]

run_id = st.text_input("Run ID — también podés pegarlo desde Swagger",
                       value=st.session_state.c14_run_id)
if not run_id:
    st.info("Primero creá una ejecución o pegá un run_id existente.")
    st.stop()
run = call_api("GET", f"/runs/{run_id}")
if not run:
    st.stop()
st.subheader("Ejecución")
st.write("**Estado:**", run["status"])
st.write("**thread_id:**", run["thread_id"])
st.code(run["run_id"], language=None)
action = run.get("pending_action")
if action:
    st.subheader("Propuesta para revisión")
    st.write("**Versión:**", action["version"])
    st.write("**Fuentes:**", action.get("evidence_used") or "Sin evidencia")
    st.text_area("Borrador", action["args"].get("body", ""), height=130, disabled=True)

if run["status"] == "waiting_for_approval" and action:
    with st.form("human_gate"):
        choice = st.radio("Decisión", ["approve", "edit", "reject"], horizontal=True)
        revised = st.text_area("Cuerpo editado (solo si elegís edit)",
                               action["args"].get("body", ""))
        submit = st.form_submit_button("Confirmar decisión")
    if submit:
        data = {"decision": choice, "actor": "alumno-c14",
                "action_id": action["action_id"],
                "proposal_version": action["version"], "args_hash": action["args_hash"]}
        if choice == "edit":
            data["edited_args"] = {"body": revised}
        if call_api("POST", f"/runs/{run_id}/decisions", json=data):
            st.rerun()

if run["status"] == "approved_pending_policy":
    if st.button("Reanudar (dry-run)", type="primary"):
        if call_api("POST", f"/runs/{run_id}/resume"):
            st.rerun()
if run.get("result"):
    st.success("Resultado guardado. No hubo envío real.")
    st.json(run["result"])
with st.expander("Eventos persistidos"):
    st.json(call_api("GET", f"/runs/{run_id}/events") or [])

### Levantar Streamlit y probar el mismo run

**Segunda terminal**, desde `ai_agent_project`:

```bash
python -m streamlit run streamlit_app.py --server.port 8501
```

Abrí **http://localhost:8501**. La API de la primera terminal debe seguir activa.

### PARADA E — Dos navegadores, un run

1. Desde Streamlit elegí un caso y creá una ejecución.
2. Copiá su `run_id`, abrí Swagger y consultá `GET /runs/{run_id}`.
3. En Streamlit aprobá o editá. Actualizá la consulta de Swagger.
4. En Swagger, cuando corresponda, reanudá el run; recargá Streamlit.

**Resultado esperado:** ambos clientes muestran el mismo estado porque consultan la misma API. Si los estados difieren, comprobá primero que los dos apunten al mismo servidor.

## 8. Docker Compose — Dejar de depender de la notebook

En Docker: una **imagen** empaqueta el software; un **contenedor** lo ejecuta; una **red interna** conecta la UI con la API (`http://api:8000`); un **volumen** conserva los checkpoints de C11 si se reemplaza un contenedor.

El corpus sintético se incluye en la imagen; el JSON de estado se guarda en un volumen separado. Para este laboratorio se configura un único worker de Uvicorn porque `JsonCheckpointStore` no coordina escrituras entre procesos diferentes.

### 8.1 Dependencias mínimas de la aplicación

El contenedor instala las bibliotecas necesarias para FastAPI, la ejecución de pruebas HTTP y Streamlit. No requiere SDK de Gmail ni credenciales para operar con el caso sintético.

In [ ]:
%%writefile ai_agent_project/requirements-class14.txt
fastapi>=0.115,<1.0
uvicorn>=0.30,<1.0
pydantic>=2.7,<3
httpx>=0.27,<1
requests>=2.31,<3
streamlit>=1.35,<2

### 8.2 Dockerfile: definir cómo se construye la imagen

`COPY src/` incorpora nuestros módulos, pero `COPY data/mock_gmail/` agrega los archivos de ejemplo. `CMD` inicia Uvicorn por defecto; Compose lo reemplazará para el servicio de interfaz.

In [ ]:
%%writefile ai_agent_project/Dockerfile.class14
FROM python:3.12-slim
WORKDIR /app
COPY requirements-class14.txt /app/
RUN pip install --no-cache-dir -r requirements-class14.txt
COPY src/ /app/src/
COPY data/mock_gmail/ /app/data/mock_gmail/
COPY streamlit_app.py /app/
ENV PYTHONPATH=/app/src C14_PROJECT_DIR=/app
RUN mkdir -p /app/runtime
# Sólo 1 worker: JsonCheckpointStore es una implementación didáctica.
CMD ["uvicorn", "ai_agent_course.class14_api:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]

### 8.3 Compose: dos servicios en una red y un volumen durable

`api` publica el puerto 8000 y persiste checkpoints en `c14_runtime`. `ui` se comunica internamente con `api:8000`, no con `localhost:8000` (dentro del contenedor `localhost` es el propio contenedor).

In [ ]:
%%writefile ai_agent_project/compose.class14.yml
services:
  api:
    build:
      context: .
      dockerfile: Dockerfile.class14
    environment:
      C14_STORE_PATH: /app/runtime/class14_runs.json
    ports:
      - "127.0.0.1:8000:8000"
    volumes:
      - c14_runtime:/app/runtime
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://localhost:8000/health/ready')"]
      interval: 10s
      timeout: 5s
      retries: 6
  ui:
    build:
      context: .
      dockerfile: Dockerfile.class14
    command: ["streamlit", "run", "streamlit_app.py", "--server.address=0.0.0.0", "--server.port=8501", "--server.headless=true"]
    environment:
      C14_API_URL: http://api:8000
    ports:
      - "127.0.0.1:8501:8501"
    depends_on:
      api:
        condition: service_healthy
volumes:
  c14_runtime:

### PARADA F — Reiniciar servicios sin perder la ejecución

**Cerrá los procesos Uvicorn y Streamlit locales para liberar los puertos 8000 y 8501.** Desde `ai_agent_project`:

```bash
docker compose -f compose.class14.yml up --build -d
docker compose -f compose.class14.yml ps
```

1. Abrí http://localhost:8000/docs o http://localhost:8501.
2. Creá un run y guardá su `run_id`.
3. Ejecutá `docker compose -f compose.class14.yml restart api`.
4. Consultá el mismo run: debería seguir allí.
5. Al terminar, ejecutá `docker compose -f compose.class14.yml down` (**sin** `-v`, para no borrar el volumen).

**Atención:** el servicio Docker usa su **propio volumen**. Los IDs de runs creados en el proceso local no tienen por qué existir en Docker: son dos almacenes diferentes.

## 9. Integración y criterios de cierre

| Verificación | Evidencia observable |
|---|---|
| Adaptador | Usa `run_supervisor_workers` (C12) y `search_support_kb` (C13) |
| HITL | `prepare_action`, `apply_human_decision` y `mark_policy_authorized` (C11) |
| HTTP | Swagger muestra `201`, `200`, `404`, `409`, `422` |
| Persistencia | Un run se recupera tras recargar el navegador y reiniciar la API |
| Eventos | JSON y SSE devuelven el mismo historial (replay) |
| UI | Swagger y Streamlit comparten el run por ID |
| Docker | Dos servicios, red interna, health check y volumen duradero |

**Lo que deliberadamente no implementamos:** Gmail real, LLM nuevo, autenticación/autorización, un worker distribuido, SSE de eventos futuros ni el grafo LangGraph compilado dentro de la notebook de C11. La ejecución utiliza **las transiciones persistidas en `hitl.py`**, no un grafo importado que todavía no existe.

## 10. Reporte JSON y continuidad con la Clase 15

A lo largo del curso cada laboratorio consolida su evidencia en `ai_agent_project/artifacts/classXX_report.json`. En esta clase recuperamos **el mismo contrato de salida**, generando `class14_report.json` a partir de las verificaciones ejecutadas: continuidad C11–C13, endpoints, HITL, eventos, persistencia y archivos de despliegue.

El reporte registra resultados **observados**, no afirma que Docker o Streamlit se hayan probado solo porque sus archivos existan. Tampoco guarda credenciales, contenido de correos ni datos privados. La Clase 15 puede utilizar este JSON como punto de partida para evaluar calidad y observabilidad.

In [ ]:
# Evidencia consolidada C14. El patrón es el mismo de las clases anteriores:
# producir artifacts/class14_report.json y verificar su lectura.
from datetime import datetime, timezone
from IPython.display import JSON, display

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Reutilizamos las respuestas HTTP y el State C11, sin duplicar su información sensible.
endpoint_contract = sorted(
    {f"{method.upper()} {path}"
     for path, config in app.openapi()["paths"].items()
     for method in config if method.lower() in {"get", "post", "put", "delete", "patch"}}
)
http_tests = {
    "health_live": client.get("/health/live").status_code == 200,
    "health_ready": client.get("/health/ready").status_code == 200,
    "cases": cases_http.status_code == 200,
    "create_run_201": created.status_code == 201,
    "read_run_200": client.get(f"/runs/{http_run_id}").status_code == 200,
    "validation_422": client.post("/runs", json={"thread_id": ""}).status_code == 422,
    "not_found_404": client.get("/runs/no-existe").status_code == 404,
    "stale_approval_409": stale_response.status_code == 409,
    "edit_versioned": new_action["version"] == first_action["version"] + 1,
    "approval_and_resume": finished_http["status"] == "completed",
    "dry_run_no_email": finished_http["result"]["email_sent"] is False,
    "events_json_sse_equal": sse_response.text.count("data: ") == len(http_events),
}
assert all(http_tests.values()), {k: v for k, v in http_tests.items() if not v}

files = {
    "fastapi_module": "src/ai_agent_course/class14_api.py",
    "application_service": "src/ai_agent_course/class14_service.py",
    "support_flow": "src/ai_agent_course/class14_support_flow.py",
    "streamlit_app": "streamlit_app.py",
    "dockerfile": "Dockerfile.class14",
    "docker_compose": "compose.class14.yml",
    "requirements": "requirements-class14.txt",
}
file_checks = {name: (PROJECT_DIR / rel).is_file() for name, rel in files.items()}
assert all(file_checks.values()), file_checks

report = {
    "class": 14,
    "stage": "api_deployment",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "project": "ai_agent_project",
    "case_study": {
        "name": "agente de gestión asistida de emails",
        "stage": "application_boundary",
        "email_sent": False,
        "mode": "synthetic_dry_run",
    },
    "source_checkpoint": {
        "c11": {"module": "hitl.py", "state": "AgentRunState", "checkpoint": "JsonCheckpointStore"},
        "c12": {"module": "multiagent.py", "runner": "run_supervisor_workers"},
        "c13": {"module": "support_mcp_bridge.py", "search": "search_support_kb"},
        "class13_report_present": (ARTIFACTS_DIR / "class13_report.json").is_file(),
        "langgraph_graph_imported": False,
    },
    "architecture": {
        "transport": "FastAPI",
        "application_service": "CourseApplication",
        "domain_state_source": "C11/hitl.py",
        "ui": "Streamlit (cliente HTTP)",
        "execution": "reused C11-C13 + local dry-run",
    },
    "api": {
        "endpoints": endpoint_contract,
        "test_result": http_tests,
        "status_codes_observed": [200, 201, 404, 409, 422],
    },
    "runtime": {
        "storage": "JsonCheckpointStore",
        "persistence_reopened": recovered.run_id == completed.run_id and recovered.status == "completed",
        "manual_workflow_status": completed.status,
        "manual_workflow_email_sent": completed.result["email_sent"],
        "hitl": {
            "initial_version": a1.version,
            "edited_version": a2.version,
            "stale_approval_rejected": http_tests["stale_approval_409"],
            "final_status": finished_http["status"],
        },
        "events": {
            "json_count": len(http_events),
            "sse_replay_count": sse_response.text.count("data: "),
            "streaming_mode": "replay_of_persisted_events",
            "live_subscription": False,
        },
    },
    "deployment": {
        "files_present": file_checks,
        "docker_compose_runtime_check": "manual_pending",
        "streamlit_browser_check": "manual_pending",
        "limits": ["single-process JSON checkpoint store", "no authentication",
                   "no Gmail send", "no live SSE subscription"],
    },
    "definition_of_done": {
        "domain_reuse": all(file_checks[k] for k in ("support_flow", "application_service")),
        "api_tests": all(http_tests.values()),
        "checkpoint_persistence": recovered.status == "completed",
        "sse_replay": http_tests["events_json_sse_equal"],
        "deployment_files": all(file_checks.values()),
        "docker_running": "manual_pending",
        "streamlit_running": "manual_pending",
    },
    "continuity": {
        "increment": "frontera FastAPI + contratos HTTP + UI Streamlit + Docker Compose",
        "next_class": 15,
        "next_problem": "evaluación, observabilidad, pruebas de regresión y quality gates",
    },
}
report_path = ARTIFACTS_DIR / "class14_report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
loaded_report = json.loads(report_path.read_text(encoding="utf-8"))
assert loaded_report["class"] == 14
assert loaded_report["continuity"]["next_class"] == 15
assert loaded_report["api"]["test_result"]["stale_approval_409"]

# Vista legible del JSON generado y handoff al laboratorio siguiente.
summary = {
    "class": loaded_report["class"],
    "report": str(report_path.relative_to(PROJECT_DIR)),
    "api_checks_ok": sum(loaded_report["api"]["test_result"].values()),
    "api_checks_total": len(loaded_report["api"]["test_result"]),
    "hitl_final": loaded_report["runtime"]["hitl"]["final_status"],
    "docker": loaded_report["deployment"]["docker_compose_runtime_check"],
    "streamlit": loaded_report["deployment"]["streamlit_browser_check"],
    "next_class": loaded_report["continuity"]["next_class"],
}
print("✓ Reporte Clase 14 generado:", report_path.relative_to(PROJECT_DIR))
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(JSON(loaded_report, expanded=False))

### Autoevaluación (responder sin mirar el código)

1. ¿Qué diferencia hay entre `thread_id` y `run_id`? ¿Qué devuelve `POST /runs`?
2. ¿Por qué no se implementa `apply_human_decision` dentro del decorador `@app.post`?
3. ¿Cuál es el significado de un 409 cuando enviás una aprobación vieja?
4. ¿Qué explica que Streamlit pueda inspeccionar un run creado en Swagger?
5. ¿Qué relación tiene `http://api:8000` con la red de Compose?
6. ¿Qué prueba que el stream SSE de la clase **no** es seguimiento en vivo?

**Puente a la Clase 15:** tenemos una aplicación accesible y verificable. El paso siguiente es medir su calidad, instrumentar su observabilidad y probar regresiones de forma sistemática.